#Module 1

**Cellule 1** : installation et imports

In [1]:
!pip -q install plotly
import pandas as pd, numpy as np
import plotly.express as px


**Cellule 2** : chargement

In [2]:
URL = "https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv"

def load_data(source=URL):
    df = pd.read_csv(source, na_values=["?"])   # "?" = valeur manquante dans ce jeu
    df = df.dropna(axis=1, how="all")           # supprime la colonne vide en fin de fichier
    return df

df = load_data()
print(df.shape)
df.head()

(1000, 39)


,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,...,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,...,0,NaN,5070,780,780,3510,Mercedes,E400,2007,Y
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,...,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,...,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,...,1,NO,6500,1300,650,4550,Accura,RSX,2009,N


**Cellule 3**

In [3]:
df.info()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_number                1000 non-null   int64  
 3   policy_bind_date             1000 non-null   object 
 4   policy_state                 1000 non-null   object 
 5   policy_csl                   1000 non-null   object 
 6   policy_deductable            1000 non-null   int64  
 7   policy_annual_premium        1000 non-null   float64
 8   umbrella_limit               1000 non-null   int64  
 9   insured_zip                  1000 non-null   int64  
 10  insured_sex                  1000 non-null   object 
 11  insured_education_level      1000 non-null   object 
 12  insured_occupation           1000 non-null   object 
 13  insured_hobbies    

,count,mean,std,min,25%,50%,75%,max
months_as_customer,1000.0,2.039540e+02,1.151132e+02,0.00,115.7500,199.5,276.250,479.00
age,1000.0,3.894800e+01,9.140287e+00,19.00,32.0000,38.0,44.000,64.00
policy_number,1000.0,5.462386e+05,2.570630e+05,100804.00,335980.2500,533135.0,759099.750,999435.00
policy_deductable,1000.0,1.136000e+03,6.118647e+02,500.00,500.0000,1000.0,2000.000,2000.00
policy_annual_premium,1000.0,1.256406e+03,2.441674e+02,433.33,1089.6075,1257.2,1415.695,2047.59
umbrella_limit,1000.0,1.101000e+06,2.297407e+06,-1000000.00,0.0000,0.0,0.000,10000000.00
insured_zip,1000.0,5.012145e+05,7.170161e+04,430104.00,448404.5000,466445.5,603251.000,620962.00
capital-gains,1000.0,2.512610e+04,2.787219e+04,0.00,0.0000,0.0,51025.000,100500.00
capital-loss,1000.0,-2.679370e+04,2.810410e+04,-111100.00,-51500.0000,-23250.0,0.000,0.00
incident_hour_of_the_day,1000.0,1.164400e+01,6.951373e+00,0.00,6.0000,12.0,17.000,23.00


**Cellule 4** : la cible

In [4]:
df["fraud"] = (df["fraud_reported"] == "Y").astype(int)
(df["fraud"].value_counts(normalize=True)*100).round(1)

,proportion
fraud,
0,75.3
1,24.7


**Cellule 5** :valeurs manquantes et proportion des classes

In [5]:
miss = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
miss[miss > 0]

,0
property_damage,36.0
police_report_available,34.3
collision_type,17.8
authorities_contacted,9.1


**Cellule 6** : graphique de la répartition des classes

In [6]:
fig = px.pie(df, names="fraud_reported", title="Répartition des classes",
             color="fraud_reported", color_discrete_map={"Y":"#d62728","N":"#1f77b4"})
fig.show()

**Cellule 7** : séparer les types de colonnes

In [8]:
TARGET = "fraud"

# Identifiants numériques : à exclure des variables numériques
id_cols = ["policy_number", "insured_zip"]
# Colonnes texte à exclure : adresse quasi-unique, dates brutes, cible d'origine
drop_obj = ["incident_location", "policy_bind_date", "incident_date", "fraud_reported"]

num_cols = df.select_dtypes("number").columns.drop([TARGET] + id_cols).tolist()
cat_cols = df.select_dtypes("object").columns.drop(drop_obj).tolist()
print(len(num_cols), "numériques |", len(cat_cols), "catégorielles")

16 numériques | 17 catégorielles


**Cellule 8** : distribution d’une variable numérique selon la classe

In [9]:
COLORS = {"Y": "#d62728", "N": "#1f77b4"}
px.histogram(df, x="total_claim_amount", color="fraud_reported", barmode="overlay",
             opacity=0.6, marginal="box", color_discrete_map=COLORS)

### Interprétation de la distribution du montant des réclamations (`total_claim_amount`)

* **Forte superposition** : La répartition des montants est très similaire entre les sinistres frauduleux (rouge) et non frauduleux (bleu). Le montant seul ne suffit pas à détecter la fraude.
* **Pic principal** : La grande majorité des réclamations (qu'il y ait fraude ou non) se concentre entre **40 000\$  et 80 000 \$**
* **Légère tendance** : Les réclamations frauduleuses (boîte rouge en haut) affichent des montants médians légèrement plus élevés et comportent très peu de dossiers à très faible coût (inférieurs à 10 000 $).

**Cellule 9** : corrélations

In [10]:
corr = df[num_cols + [TARGET]].corr().round(2)
px.imshow(corr, text_auto=True, aspect="auto",
          color_continuous_scale="RdBu_r", zmin=-1, zmax=1)

In [11]:
corr[TARGET].drop(TARGET).sort_values()

,fraud
capital-gains,-0.02
policy_annual_premium,-0.01
capital-loss,-0.01
incident_hour_of_the_day,0.00
age,0.01
policy_deductable,0.01
auto_year,0.01
months_as_customer,0.02
bodily_injuries,0.03
witnesses,0.05


**Cellule 10** : taux de fraude par variable (graphique exigé par le sujet)

In [12]:
def fraud_rate(df, col):
    t = (df.assign(**{col: df[col].astype(str).replace("nan", "Manquant")})
           .groupby(col)[TARGET].agg(taux="mean", n="size").reset_index()
           .sort_values("taux", ascending=False))
    t["taux"] = (t["taux"] * 100).round(1)
    return t

def plot_fraud_rate(col):
    t = fraud_rate(df, col)
    fig = px.bar(t, x=col, y="taux", text="n", title=f"Taux de fraude (%) par {col}")
    fig.add_hline(y=df[TARGET].mean() * 100, line_dash="dash",
                  annotation_text="taux global")
    fig.update_traces(textposition="outside")
    fig.show()

for c in ["incident_severity", "insured_hobbies", "collision_type", "police_report_available"]:
    plot_fraud_rate(c)

#Module 2 : préparation

**Cellule 11** : créer des variables à partir des dates

In [13]:
d = pd.to_datetime(df["incident_date"])
b = pd.to_datetime(df["policy_bind_date"])
df["policy_age_days"] = (d - b).dt.days      # ancienneté de la police le jour du sinistre
df["incident_dow"] = d.dt.dayofweek          # 0 = lundi

**Cellule 12** : manquants et redondance

In [16]:
MISSING_COLS = ["collision_type", "property_damage",
                "police_report_available", "authorities_contacted"]
df[MISSING_COLS] = df[MISSING_COLS].fillna("Manquant")

# Le total est-il bien la somme des trois montants ?
(df["total_claim_amount"] ==
 df["injury_claim"] + df["property_claim"] + df["vehicle_claim"]).mean()


np.float64(1.0)

**Cellule 13** : liste des variables et variables sensibles

In [17]:
num_feats = [c for c in num_cols if c != "total_claim_amount"] + ["policy_age_days", "incident_dow"]
cat_feats = cat_cols
FEATURES = num_feats + cat_feats

# À garder identifiables pour l'analyse d'équité du one-pager
SENSITIVE = ["insured_sex", "age", "insured_education_level",
             "insured_occupation", "insured_relationship"]

**Cellule 14** : split paramétrable et préprocesseur

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_split(df, features, test_size=0.2, seed=42):
    return train_test_split(df[features], df[TARGET], test_size=test_size,
                            stratify=df[TARGET], random_state=seed)

def make_preprocessor(num, cat, scale=True):
    return ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])

X_train, X_test, y_train, y_test = make_split(df, FEATURES)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

(800, 34) (200, 34) 0.248 0.245


#Module 3 : modélisation et comparaison

**Cellule 15** : définir les modèles

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

pos_w = (y_train == 0).sum() / (y_train == 1).sum()   # poids de la classe fraude

# (estimateur, normaliser ?)
MODELS = {
    "Régression logistique": (LogisticRegression(max_iter=2000, class_weight="balanced"), True),
    "Arbre de décision":     (DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42), False),
    "KNN":                   (KNeighborsClassifier(n_neighbors=15), True),
    "Random Forest":         (RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42), False),
    "XGBoost":               (XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                                            scale_pos_weight=pos_w, eval_metric="logloss",
                                            random_state=42), False),
}

def build_pipe(name):
    est, scale = MODELS[name]
    return Pipeline([("prep", make_preprocessor(num_feats, cat_feats, scale=scale)),
                     ("clf", est)])

**Cellule 16** : tableau comparatif en validation croisée

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["roc_auc", "average_precision", "f1", "recall", "precision"]

rows = []
for name in MODELS:
    r = cross_validate(build_pipe(name), X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"Modèle": name,
                 **{k.replace("test_", ""): r[k].mean().round(3) for k in r if k.startswith("test_")}})

results = pd.DataFrame(rows).sort_values("average_precision", ascending=False)
results

,Modèle,roc_auc,average_precision,f1,recall,precision
4,XGBoost,0.861,0.678,0.702,0.783,0.639
0,Régression logistique,0.868,0.670,0.695,0.783,0.628
3,Random Forest,0.846,0.597,0.346,0.243,0.609
1,Arbre de décision,0.784,0.581,0.698,0.778,0.636
2,KNN,0.624,0.330,0.045,0.025,0.217


**Cellule 17** : règle de base

In [22]:
from sklearn.metrics import f1_score, recall_score, precision_score
base = (X_train["incident_severity"] == "Major Damage").astype(int)
print("F1", round(f1_score(y_train, base), 3),
      "| rappel", round(recall_score(y_train, base), 3),
      "| précision", round(precision_score(y_train, base), 3))

F1 0.628 | rappel 0.662 | précision 0.598


#Optimisation des hyperparamètres

**Cellule 18** : espaces de recherche

In [23]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform, uniform

PARAMS = {
    "Régression logistique": {"clf__C": loguniform(1e-3, 1e2)},
    "Arbre de décision": {"clf__max_depth": randint(2, 10),
                          "clf__min_samples_leaf": randint(5, 40)},
    "Random Forest": {"clf__n_estimators": randint(200, 600),
                      "clf__max_depth": randint(3, 12),
                      "clf__min_samples_leaf": randint(1, 15),
                      "clf__max_features": ["sqrt", 0.3, 0.5]},
    "XGBoost": {"clf__n_estimators": randint(100, 500),
                "clf__max_depth": randint(2, 6),
                "clf__learning_rate": loguniform(0.01, 0.2),
                "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.5, 0.5),
                "clf__min_child_weight": randint(1, 10)},
}

def tune(name, n_iter=30):
    s = RandomizedSearchCV(build_pipe(name), PARAMS[name], n_iter=n_iter,
                           scoring="average_precision", cv=cv, n_jobs=-1, random_state=42)
    return s.fit(X_train, y_train)

**Cellule 19** : lancer la recherche

In [24]:
tuned = {}
for name in PARAMS:
    tuned[name] = tune(name)
    print(f"{name:22s} PR-AUC CV = {tuned[name].best_score_:.3f}")
    print("   ", tuned[name].best_params_)

Régression logistique  PR-AUC CV = 0.670
    {'clf__C': np.float64(1.0129197956845732)}
Arbre de décision      PR-AUC CV = 0.674
    {'clf__max_depth': 3, 'clf__min_samples_leaf': 6}
Random Forest          PR-AUC CV = 0.673
    {'clf__max_depth': 10, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 3, 'clf__n_estimators': 488}
XGBoost                PR-AUC CV = 0.677
    {'clf__colsample_bytree': np.float64(0.954660201039391), 'clf__learning_rate': np.float64(0.02171103454376615), 'clf__max_depth': 5, 'clf__min_child_weight': 4, 'clf__n_estimators': 101, 'clf__subsample': np.float64(0.7700623497964979)}


#Module 4 : évaluation sur le test

**Cellule 20** : prédictions et tableau des métriques

In [25]:
import plotly.graph_objects as go
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, recall_score,
                             precision_score, roc_curve, precision_recall_curve, confusion_matrix)

best = {n: tuned[n].best_estimator_ for n in tuned}        # déjà ré-entraînés sur tout le train
best["KNN"] = build_pipe("KNN").fit(X_train, y_train)
proba = {n: m.predict_proba(X_test)[:, 1] for n, m in best.items()}

def metrics_at(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    return {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "Précision": precision_score(y, pred, zero_division=0),
            "Rappel": recall_score(y, pred), "F1": f1_score(y, pred)}

pd.DataFrame({n: metrics_at(y_test, p) for n, p in proba.items()}).T.round(3).sort_values("PR-AUC", ascending=False)

,ROC-AUC,PR-AUC,Précision,Rappel,F1
XGBoost,0.842,0.618,0.636,0.857,0.730
Arbre de décision,0.848,0.595,0.636,0.857,0.730
Random Forest,0.825,0.588,0.635,0.816,0.714
Régression logistique,0.826,0.529,0.618,0.694,0.654
KNN,0.611,0.303,0.333,0.020,0.038


**Cellule 21** : courbes ROC superposées et courbes précision-rappel

In [26]:
fig = go.Figure()
for n, p in proba.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    fig.add_scatter(x=fpr, y=tpr, name=f"{n} (AUC={roc_auc_score(y_test, p):.2f})")
fig.add_scatter(x=[0, 1], y=[0, 1], line=dict(dash="dash"), name="Hasard")
fig.update_layout(title="Courbes ROC", xaxis_title="Taux de faux positifs", yaxis_title="Rappel")
fig.show()

fig = go.Figure()
for n, p in proba.items():
    pr, rc, _ = precision_recall_curve(y_test, p)
    fig.add_scatter(x=rc, y=pr, name=f"{n} (AP={average_precision_score(y_test, p):.2f})")
fig.add_hline(y=y_test.mean(), line_dash="dash", annotation_text="hasard")
fig.update_layout(title="Courbes précision-rappel", xaxis_title="Rappel", yaxis_title="Précision")
fig.show()

**Cellule 22** : matrice de confusion avec curseur de seuil

In [27]:
from ipywidgets import interact, FloatSlider, Dropdown

def show_threshold(name="XGBoost", thr=0.5):
    p = proba[name]
    pred = (p >= thr).astype(int)
    m = metrics_at(y_test, p, thr)
    print(f"{name} | seuil {thr:.2f} → rappel {m['Rappel']:.2f} | précision {m['Précision']:.2f} | F1 {m['F1']:.2f}")
    px.imshow(confusion_matrix(y_test, pred), text_auto=True, color_continuous_scale="Blues",
              x=["Prédit légitime", "Prédit fraude"], y=["Réel légitime", "Réel fraude"]).show()

interact(show_threshold, name=Dropdown(options=list(proba), value="XGBoost"),
         thr=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5));

interactive(children=(Dropdown(description='name', index=3, options=('Régression logistique', 'Arbre de décisi…

In [28]:
base_te = (X_test["incident_severity"] == "Major Damage").astype(int)
pd.Series(metrics_at(y_test, base_te.to_numpy(float))).round(3)

,0
ROC-AUC,0.798
PR-AUC,0.529
Précision,0.632
Rappel,0.735
F1,0.679


#Module 6 : API FastAPI.

sauvegarder les modèles entraînés

**Cellule 28**

In [29]:
import joblib
joblib.dump({"models": best, "features": FEATURES}, "bundle.joblib")
print(list(best), "|", len(FEATURES), "variables")

['Régression logistique', 'Arbre de décision', 'Random Forest', 'XGBoost', 'KNN'] | 34 variables


écrire l’API

**Cellule 29**

In [30]:
%%writefile api.py
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

BUNDLE = joblib.load("bundle.joblib")
MODELS, FEATURES = BUNDLE["models"], BUNDLE["features"]

app = FastAPI(
    title="API de détection de fraude auto",
    description="Prédit la probabilité qu'un dossier de sinistre automobile soit frauduleux.",
    version="1.0",
)

class PredictRequest(BaseModel):
    modele: str = Field("XGBoost", description="Nom du modèle (voir GET /models)")
    seuil: float = Field(0.5, ge=0, le=1, description="Seuil de décision")
    variables: dict = Field(..., description="Valeurs des variables explicatives du dossier")

class PredictResponse(BaseModel):
    modele: str
    probabilite_fraude: float
    seuil: float
    decision: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/models")
def list_models():
    return {"modeles": list(MODELS), "variables_attendues": FEATURES}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    if req.modele not in MODELS:
        raise HTTPException(404, f"Modèle inconnu : {req.modele}")
    missing = [f for f in FEATURES if f not in req.variables]
    if missing:
        raise HTTPException(422, f"Variables manquantes : {missing}")
    X = pd.DataFrame([req.variables])[FEATURES]
    p = float(MODELS[req.modele].predict_proba(X)[0, 1])
    return PredictResponse(
        modele=req.modele,
        probabilite_fraude=round(p, 4),
        seuil=req.seuil,
        decision="À investiguer" if p >= req.seuil else "Traitement standard",
    )

Writing api.py


**Cellule 30**:lancer l’API dans Colab

In [54]:
!pip -q install fastapi uvicorn
import subprocess, time, requests

proc = subprocess.Popen(["uvicorn", "api:app", "--port", "8000"])
time.sleep(6)
print(requests.get("http://127.0.0.1:8000/health").json())
print(requests.get("http://127.0.0.1:8000/models").json()["modeles"])

{'status': 'ok'}
['Régression logistique', 'Arbre de décision', 'Random Forest', 'XGBoost', 'KNN']


**Cellule 31** :tester une prédiction

In [32]:
import json

row = json.loads(X_test.iloc[[0]].to_json(orient="records"))[0]   # types JSON propres
payload = {"modele": "XGBoost", "seuil": 0.5, "variables": row}

r = requests.post("http://127.0.0.1:8000/predict", json=payload)
print(r.status_code, r.json())
print("Vérité terrain :", int(y_test.iloc[0]))

200 {'modele': 'XGBoost', 'probabilite_fraude': 0.1011, 'seuil': 0.5, 'decision': 'Traitement standard'}
Vérité terrain : 0


In [33]:
bad = {**payload, "modele": "Inconnu"}
print(requests.post("http://127.0.0.1:8000/predict", json=bad).status_code)        # attendu : 404
bad = {**payload, "variables": {"age": 40}}
print(requests.post("http://127.0.0.1:8000/predict", json=bad).status_code)        # attendu : 422

404
422


**Cellule 32** : core.py

In [34]:
%%writefile core.py
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv"
DEFAULT_TARGET, DEFAULT_POSITIVE = "fraud_reported", "Y"
MISSING_COLS = ["collision_type", "property_damage", "police_report_available", "authorities_contacted"]
SENSITIVE = ["insured_sex", "age", "insured_education_level", "insured_occupation", "insured_relationship"]

def load_data(source=URL):
    return pd.read_csv(source, na_values=["?"]).dropna(axis=1, how="all")

def add_target(df, target, positive):
    df = df.copy()
    df["target"] = (df[target].astype(str) == str(positive)).astype(int)
    return df

def make_split(df, features, test_size=0.2, seed=42):
    return train_test_split(df[features], df["target"], test_size=test_size,
                            stratify=df["target"], random_state=seed)

def make_preprocessor(num, cat, scale=True):
    return ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])

Writing core.py


**Cellule 33** : requirements.txt

In [35]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
xgboost
plotly
joblib
requests
fastapi
uvicorn

Writing requirements.txt


**Cellule 34** : app.py avec le Module 1

In [36]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
from core import *

st.set_page_config(page_title="Fraude auto : ML explicable", layout="wide")
st.title("Détection de fraude aux sinistres auto")

@st.cache_data
def get_data(file):
    return load_data(file) if file is not None else load_data()

# ---------- Barre latérale : source de données et cible ----------
with st.sidebar:
    st.header("Données")
    up = st.file_uploader("Importer un CSV (sinon : jeu de données par défaut)", type="csv")
    raw = get_data(up)
    cols = list(raw.columns)
    default_t = cols.index(DEFAULT_TARGET) if DEFAULT_TARGET in cols else len(cols) - 1
    target = st.selectbox("Variable cible", cols, index=default_t)
    values = sorted(raw[target].dropna().astype(str).unique())
    default_p = values.index(DEFAULT_POSITIVE) if DEFAULT_POSITIVE in values else 0
    positive = st.selectbox("Classe positive", values, index=default_p)

df = add_target(raw, target, positive)
num_all = df.select_dtypes("number").columns.drop("target").tolist()
cat_all = df.select_dtypes("object").columns.drop(target, errors="ignore").tolist()

tab1, = st.tabs(["1 · Données"])

with tab1:
    c1, c2, c3 = st.columns(3)
    c1.metric("Lignes", f"{len(df):,}")
    c2.metric("Colonnes", df.shape[1] - 1)
    c3.metric("Taux de classe positive", f"{df['target'].mean():.1%}")

    st.subheader("Aperçu")
    st.dataframe(df.head(20), use_container_width=True)
    st.subheader("Statistiques descriptives")
    st.dataframe(df[num_all].describe().T, use_container_width=True)

    left, right = st.columns(2)
    with left:
        st.subheader("Valeurs manquantes")
        miss = (df.isna().mean() * 100).round(1)
        miss = miss[miss > 0].sort_values(ascending=False)
        st.dataframe(miss.rename("% manquant")) if len(miss) else st.write("Aucune")
    with right:
        st.subheader("Répartition des classes")
        st.plotly_chart(px.pie(df, names=target), use_container_width=True)

    st.subheader("Distribution d'une variable")
    var = st.selectbox("Variable numérique", num_all)
    st.plotly_chart(px.histogram(df, x=var, color=target, barmode="overlay",
                                 opacity=0.6, marginal="box"), use_container_width=True)

    st.subheader("Corrélations")
    st.plotly_chart(px.imshow(df[num_all + ["target"]].corr().round(2), text_auto=True, aspect="auto",
                              color_continuous_scale="RdBu_r", zmin=-1, zmax=1),
                    use_container_width=True)

    st.subheader("Taux de la classe positive par variable")
    col = st.selectbox("Variable catégorielle", cat_all)
    t = (df.assign(**{col: df[col].fillna("Manquant").astype(str)})
           .groupby(col)["target"].agg(taux="mean", n="size").reset_index()
           .sort_values("taux", ascending=False))
    t["taux"] = (t["taux"] * 100).round(1)
    fig = px.bar(t, x=col, y="taux", text="n")
    fig.add_hline(y=df["target"].mean() * 100, line_dash="dash", annotation_text="taux global")
    st.plotly_chart(fig, use_container_width=True)

Writing app.py


#Lancer l’application depuis Colab

**Cellule 35**

In [37]:
!pip -q install streamlit
!streamlit run app.py --server.port 8501 &>/content/streamlit.log &
!curl -s ipv4.icanhazip.com
!npx -y localtunnel --port 8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 65.2 MB/s eta 0:00:00
136.107.206.57
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://fruity-trains-stop.loca.lt
^C


In [38]:
# 1. Arrêter les anciens processus
!pkill -f streamlit
!pkill -f localtunnel

# 2. Relancer Streamlit avec les options adaptées à un tunnel
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

# 3. Installer et lancer cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1 &

# 4. Afficher le lien
import time; time.sleep(10)
!grep -o "https://[a-z0-9-]*\.trycloudflare.com" cf.log | head -1

https://disposal-humanitarian-andreas-reasoning.trycloudflare.com


**Cellule 36** : core.py

In [39]:
%%writefile core.py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

URL = "https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv"
DEFAULT_TARGET, DEFAULT_POSITIVE = "fraud_reported", "Y"
ID_COLS = ["policy_number", "insured_zip"]
RAW_DROP = ["incident_location", "policy_bind_date", "incident_date"]
REDUNDANT = ["total_claim_amount"]          # somme des trois autres montants
SENSITIVE = ["insured_sex", "age", "insured_education_level", "insured_occupation", "insured_relationship"]
SCALERS = {"Standard (z-score)": StandardScaler, "Min-Max": MinMaxScaler, "Aucune": None}

def load_data(source=URL):
    return pd.read_csv(source, na_values=["?"]).dropna(axis=1, how="all")

def add_target(df, target, positive):
    df = df.copy()
    df["target"] = (df[target].astype(str) == str(positive)).astype(int)
    return df

def engineer_features(df):
    """Variables dérivées des dates (uniquement si les colonnes existent)."""
    df = df.copy()
    if {"incident_date", "policy_bind_date"} <= set(df.columns):
        d = pd.to_datetime(df["incident_date"], errors="coerce")
        b = pd.to_datetime(df["policy_bind_date"], errors="coerce")
        df["policy_age_days"] = (d - b).dt.days
        df["incident_dow"] = d.dt.dayofweek
    return df

def split_columns(df, target):
    """Colonnes numériques / catégorielles utilisables (hors cible, identifiants, dates brutes)."""
    drop = set(ID_COLS + RAW_DROP + ["target", target])
    drop |= {c for c in df.columns if df[c].nunique() == len(df)}      # identifiants strictement uniques
    num = [c for c in df.select_dtypes("number").columns if c not in drop]
    cat = [c for c in df.select_dtypes(exclude="number").columns if c not in drop]
    return num, cat

def make_split(df, features, test_size=0.2, seed=42):
    return train_test_split(df[features], df["target"], test_size=test_size,
                            stratify=df["target"], random_state=seed)

def make_preprocessor(num, cat, scaler="Standard (z-score)", num_impute="median"):
    steps = [("imp", SimpleImputer(strategy=num_impute))]
    if SCALERS[scaler] is not None:
        steps.append(("sc", SCALERS[scaler]()))
    return ColumnTransformer([
        ("num", Pipeline(steps), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])

Overwriting core.py


**Cellule 37** : app.py

In [40]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
from core import *

st.set_page_config(page_title="Fraude auto : ML explicable", layout="wide")
st.title("Détection de fraude aux sinistres auto")

@st.cache_data
def get_data(file):
    return load_data(file) if file is not None else load_data()

# ---------- Barre latérale : source de données et cible ----------
with st.sidebar:
    st.header("Données")
    up = st.file_uploader("Importer un CSV (sinon : jeu de données par défaut)", type="csv")
    raw = get_data(up)
    cols = list(raw.columns)
    default_t = cols.index(DEFAULT_TARGET) if DEFAULT_TARGET in cols else len(cols) - 1
    target = st.selectbox("Variable cible", cols, index=default_t)
    values = sorted(raw[target].dropna().astype(str).unique())
    default_p = values.index(DEFAULT_POSITIVE) if DEFAULT_POSITIVE in values else 0
    positive = st.selectbox("Classe positive", values, index=default_p)

df = add_target(raw, target, positive)
num_all, cat_all = split_columns(df, target)

tab1, tab2 = st.tabs(["1 · Données", "2 · Préparation"])

with tab1:
    c1, c2, c3 = st.columns(3)
    c1.metric("Lignes", f"{len(df):,}")
    c2.metric("Colonnes", df.shape[1] - 1)
    c3.metric("Taux de classe positive", f"{df['target'].mean():.1%}")

    st.subheader("Aperçu")
    st.dataframe(df.head(20), width="stretch")
    st.subheader("Statistiques descriptives")
    st.dataframe(df[num_all].describe().T, width="stretch")

    left, right = st.columns(2)
    with left:
        st.subheader("Valeurs manquantes")
        miss = (df.isna().mean() * 100).round(1)
        miss = miss[miss > 0].sort_values(ascending=False)
        st.dataframe(miss.rename("% manquant")) if len(miss) else st.write("Aucune")
    with right:
        st.subheader("Répartition des classes")
        st.plotly_chart(px.pie(df, names=target), width="stretch")

    st.subheader("Distribution d'une variable")
    var = st.selectbox("Variable numérique", num_all)
    st.plotly_chart(px.histogram(df, x=var, color=target, barmode="overlay",
                                 opacity=0.6, marginal="box"), width="stretch")

    st.subheader("Corrélations")
    st.plotly_chart(px.imshow(df[num_all + ["target"]].corr().round(2), text_auto=True, aspect="auto",
                              color_continuous_scale="RdBu_r", zmin=-1, zmax=1),
                    width="stretch")

    st.subheader("Taux de la classe positive par variable")
    col = st.selectbox("Variable catégorielle", cat_all)
    t = (df.assign(**{col: df[col].fillna("Manquant").astype(str)})
           .groupby(col)["target"].agg(taux="mean", n="size").reset_index()
           .sort_values("taux", ascending=False))
    t["taux"] = (t["taux"] * 100).round(1)
    fig = px.bar(t, x=col, y="taux", text="n")
    fig.add_hline(y=df["target"].mean() * 100, line_dash="dash", annotation_text="taux global")
    st.plotly_chart(fig, width="stretch")

with tab2:
    df2 = engineer_features(df)
    num2, cat2 = split_columns(df2, target)
    options = num2 + cat2

    st.subheader("Variables explicatives")
    feats = st.multiselect("Variables à utiliser", options,
                           default=[c for c in options if c not in REDUNDANT])
    if any(c in feats for c in REDUNDANT):
        st.warning("`total_claim_amount` est la somme des trois autres montants : "
                   "la garder rend les coefficients instables.")
    sens = [c for c in SENSITIVE if c in feats]
    if sens:
        st.caption("Variables sensibles incluses (à surveiller pour l'équité) : " + ", ".join(sens))
    if not feats:
        st.stop()

    st.subheader("Traitements")
    a, b, c = st.columns(3)
    num_imp = a.selectbox("Imputation numérique", ["median", "mean", "most_frequent"])
    scaler = b.selectbox("Normalisation (régression logistique, KNN)", list(SCALERS))
    test_size = c.slider("Part du jeu de test", 0.10, 0.40, 0.20, 0.05)
    seed = st.number_input("Graine aléatoire", value=42, step=1)
    st.caption("Valeurs catégorielles manquantes → modalité « Manquant » (informative dans ce jeu de données). "
               "Encodage : one-hot.")

    num_f = [c for c in feats if c in num2]
    cat_f = [c for c in feats if c in cat2]
    d2 = df2.copy()
    d2[cat_f] = d2[cat_f].astype(object).fillna("Manquant")

    X_train, X_test, y_train, y_test = make_split(d2, feats, test_size, int(seed))
    prep = make_preprocessor(num_f, cat_f, scaler="Aucune", num_impute=num_imp).fit(X_train)
    names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]

    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Train", f"{len(X_train)} lignes")
    m2.metric("Test", f"{len(X_test)} lignes")
    m3.metric("Positifs train / test", f"{y_train.mean():.1%} / {y_test.mean():.1%}")
    m4.metric("Colonnes après encodage", len(names))

    st.subheader("Aperçu des données transformées")
    st.dataframe(pd.DataFrame(prep.transform(X_train.head(5)), columns=names), width="stretch")

    st.session_state["prep"] = dict(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
                                    num=num_f, cat=cat_f, features=feats, scaler=scaler,
                                    num_imp=num_imp, target=target, positive=positive)
    st.success("Préparation enregistrée : elle sera utilisée par l'onglet Modélisation.")

Overwriting app.py


In [41]:
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

**Cellule 39** : ajouter les modèles à core.py

In [42]:
%%writefile -a core.py


# ======================= Module 3 : modèles =======================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from xgboost import XGBClassifier
from scipy.stats import randint, loguniform, uniform

MODEL_NAMES = ["Régression logistique", "Arbre de décision", "KNN", "Random Forest", "XGBoost"]
SCALED = {"Régression logistique", "KNN"}          # modèles sensibles à l'échelle

# Réglages manuels : (libellé, plage ou liste de valeurs, défaut)
HP_UI = {
    "Régression logistique": {"C": ("C (inverse de la régularisation)", [0.001, 0.01, 0.1, 1.0, 10.0, 100.0], 1.0)},
    "Arbre de décision": {"max_depth": ("Profondeur max", (2, 12), 3),
                          "min_samples_leaf": ("Taille min. d'une feuille", (1, 40), 6)},
    "KNN": {"n_neighbors": ("Nombre de voisins", (3, 51), 15)},
    "Random Forest": {"n_estimators": ("Nombre d'arbres", (100, 600), 300),
                      "max_depth": ("Profondeur max", (3, 15), 8),
                      "min_samples_leaf": ("Taille min. d'une feuille", (1, 15), 3)},
    "XGBoost": {"n_estimators": ("Nombre d'arbres", (50, 500), 150),
                "max_depth": ("Profondeur max", (2, 8), 3),
                "learning_rate": ("Taux d'apprentissage", (0.01, 0.30), 0.05)},
}

PARAM_GRIDS = {
    "Régression logistique": {"clf__C": loguniform(1e-3, 1e2)},
    "Arbre de décision": {"clf__max_depth": randint(2, 10), "clf__min_samples_leaf": randint(5, 40)},
    "KNN": {"clf__n_neighbors": randint(3, 51)},
    "Random Forest": {"clf__n_estimators": randint(200, 600), "clf__max_depth": randint(3, 12),
                      "clf__min_samples_leaf": randint(1, 15), "clf__max_features": ["sqrt", 0.3, 0.5]},
    "XGBoost": {"clf__n_estimators": randint(100, 500), "clf__max_depth": randint(2, 6),
                "clf__learning_rate": loguniform(0.01, 0.2), "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.5, 0.5), "clf__min_child_weight": randint(1, 10)},
}

def make_estimator(name, params, pos_w=1.0):
    params = dict(params or {})
    if name == "Régression logistique":
        return LogisticRegression(max_iter=2000, class_weight="balanced", **params)
    if name == "Arbre de décision":
        return DecisionTreeClassifier(class_weight="balanced", random_state=42, **params)
    if name == "KNN":
        return KNeighborsClassifier(**params)
    if name == "Random Forest":
        return RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1, **params)
    return XGBClassifier(scale_pos_weight=pos_w, eval_metric="logloss", random_state=42, **params)

def build_pipe(name, cfg, params=None, pos_w=1.0):
    scaler = cfg["scaler"] if name in SCALED else "Aucune"
    prep = make_preprocessor(cfg["num"], cfg["cat"], scaler=scaler, num_impute=cfg["num_imp"])
    return Pipeline([("prep", prep), ("clf", make_estimator(name, params, pos_w))])

SCORING = ["roc_auc", "average_precision", "f1", "recall", "precision"]

def evaluate_cv(pipe, X, y, folds=5, seed=42):
    cv = StratifiedKFold(folds, shuffle=True, random_state=seed)
    r = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    return {k.replace("test_", ""): float(r[k].mean()) for k in r if k.startswith("test_")}

def tune(name, cfg, X, y, n_iter=15, folds=5, seed=42, pos_w=1.0):
    cv = StratifiedKFold(folds, shuffle=True, random_state=seed)
    s = RandomizedSearchCV(build_pipe(name, cfg, None, pos_w), PARAM_GRIDS[name], n_iter=n_iter,
                           scoring="average_precision", cv=cv, n_jobs=-1, random_state=seed)
    return s.fit(X, y)

Appending to core.py


**Cellule 40** : déclarer le troisième onglet dans app.py

In [43]:
s = open("app.py").read()
s = s.replace('tab1, tab2 = st.tabs(["1 · Données", "2 · Préparation"])',
              'tab1, tab2, tab3 = st.tabs(["1 · Données", "2 · Préparation", "3 · Modélisation"])')
open("app.py", "w").write(s)
print('tab3' in s)

True


**Cellule 41** : ajouter l’onglet Modélisation à app.py

In [44]:
%%writefile -a app.py

with tab3:
    P = st.session_state["prep"]
    Xtr, ytr = P["X_train"], P["y_train"]
    pos_w = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))

    st.subheader("Choix des modèles")
    chosen = st.multiselect("Modèles à entraîner", MODEL_NAMES, default=MODEL_NAMES)
    mode = st.radio("Hyperparamètres", ["Réglage manuel", "Optimisation automatique"], horizontal=True)

    manual = {}
    if mode == "Réglage manuel":
        for name in chosen:
            with st.expander(f"Hyperparamètres : {name}"):
                manual[name] = {}
                for p, (label, rng, default) in HP_UI[name].items():
                    key = f"hp_{name}_{p}"
                    if isinstance(rng, list):
                        manual[name][p] = st.select_slider(label, options=rng, value=default, key=key)
                    elif isinstance(default, float):
                        manual[name][p] = st.slider(label, float(rng[0]), float(rng[1]), default, key=key)
                    else:
                        manual[name][p] = st.slider(label, rng[0], rng[1], default, key=key)
        n_iter = 15
    else:
        n_iter = st.slider("Nombre de combinaisons testées par modèle", 5, 40, 15)
        st.caption("Recherche aléatoire, critère : PR-AUC en validation croisée sur le jeu d'entraînement.")
    folds = st.slider("Plis de validation croisée", 3, 10, 5)

    if st.button("Entraîner et comparer", type="primary", key="train"):
        if not chosen:
            st.error("Choisis au moins un modèle.")
        else:
            rows, fitted, best_params = [], {}, {}
            bar = st.progress(0.0)
            for i, name in enumerate(chosen):
                with st.spinner(f"{name}…"):
                    if mode == "Optimisation automatique":
                        s = tune(name, P, Xtr, ytr, n_iter=n_iter, folds=folds, pos_w=pos_w)
                        fitted[name] = s.best_estimator_
                        best_params[name] = {k.replace("clf__", ""): (round(v, 4) if isinstance(v, float) else v)
                                             for k, v in s.best_params_.items()}
                        m = evaluate_cv(s.best_estimator_, Xtr, ytr, folds)
                    else:
                        pipe = build_pipe(name, P, manual[name], pos_w)
                        m = evaluate_cv(pipe, Xtr, ytr, folds)
                        fitted[name] = pipe.fit(Xtr, ytr)
                        best_params[name] = manual[name]
                rows.append({"Modèle": name, "ROC-AUC": m["roc_auc"], "PR-AUC": m["average_precision"],
                             "F1": m["f1"], "Rappel": m["recall"], "Précision": m["precision"]})
                bar.progress((i + 1) / len(chosen))
            table = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
            st.session_state["models"] = dict(fitted=fitted, table=table, params=best_params, mode=mode)

    if "models" in st.session_state:
        R = st.session_state["models"]
        st.subheader("Tableau comparatif (validation croisée sur le train)")
        st.dataframe(R["table"].round(3), width="stretch", hide_index=True)
        fig = px.bar(R["table"], x="Modèle", y="PR-AUC", text=R["table"]["PR-AUC"].round(3))
        fig.add_hline(y=float(ytr.mean()), line_dash="dash", annotation_text="hasard (taux de positifs)")
        st.plotly_chart(fig, width="stretch")
        with st.expander("Hyperparamètres retenus"):
            st.json(R["params"])
        st.caption("Écarts de 0,01 à 0,02 entre modèles : à considérer comme du bruit avec un petit jeu de données. "
                   "Le jeu de test n'est utilisé qu'à l'onglet Évaluation.")

Appending to app.py


**Cellule 42** : relancer Streamlit

In [45]:
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

**Cellule 43** : ajouter la signature à core.py

In [46]:
%%writefile -a core.py


def make_sig(P):
    """Signature de la préparation : permet de détecter des modèles entraînés avec d'autres réglages."""
    return (tuple(P["features"]), tuple(P["X_test"].index), P["scaler"], P["num_imp"])

Appending to core.py


**Cellule 44** : modifier app.py (4 onglets et garde-fou)

In [47]:
s = open("app.py").read()

a = 'tab1, tab2, tab3 = st.tabs(["1 · Données", "2 · Préparation", "3 · Modélisation"])'
b = 'tab1, tab2, tab3, tab4 = st.tabs(["1 · Données", "2 · Préparation", "3 · Modélisation", "4 · Évaluation"])'
assert a in s; s = s.replace(a, b)

a = 'st.session_state["models"] = dict(fitted=fitted, table=table, params=best_params, mode=mode)'
b = 'st.session_state["models"] = dict(fitted=fitted, table=table, params=best_params, mode=mode, sig=make_sig(P))'
assert a in s; s = s.replace(a, b)

a = '        R = st.session_state["models"]\n        st.subheader("Tableau comparatif (validation croisée sur le train)")'
b = ('        R = st.session_state["models"]\n'
     '        if R.get("sig") != make_sig(P):\n'
     '            st.warning("Les réglages de préparation ont changé depuis cet entraînement : relance « Entraîner et comparer ».")\n'
     '        st.subheader("Tableau comparatif (validation croisée sur le train)")')
assert a in s; s = s.replace(a, b)

open("app.py", "w").write(s)
print("ok")

ok


**Cellule 45** : ajouter l’onglet Évaluation

In [48]:
%%writefile -a app.py

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, recall_score,
                             precision_score, roc_curve, precision_recall_curve, confusion_matrix)

with tab4:
    if "models" not in st.session_state:
        st.info("Entraîne d'abord les modèles dans l'onglet 3 · Modélisation.")
    elif st.session_state["models"].get("sig") != make_sig(st.session_state["prep"]):
        st.warning("Les réglages de préparation ont changé depuis l'entraînement : "
                   "relance « Entraîner et comparer » dans l'onglet 3 avant d'évaluer.")
    else:
        P, R = st.session_state["prep"], st.session_state["models"]
        Xte, yte = P["X_test"], P["y_test"].to_numpy()
        proba = {n: m.predict_proba(Xte)[:, 1] for n, m in R["fitted"].items()}

        thr = st.slider("Seuil de décision", 0.05, 0.95, 0.50, 0.05, key="thr")

        rows = []
        for n, p in proba.items():
            pred = (p >= thr).astype(int)
            rows.append({"Modèle": n, "ROC-AUC": roc_auc_score(yte, p), "PR-AUC": average_precision_score(yte, p),
                         "Précision": precision_score(yte, pred, zero_division=0),
                         "Rappel": recall_score(yte, pred, zero_division=0),
                         "F1": f1_score(yte, pred, zero_division=0)})
        st.subheader(f"Métriques sur le jeu de test (seuil = {thr:.2f})")
        st.dataframe(pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).round(3),
                     hide_index=True, width="stretch")
        st.caption(f"Le test contient {int(yte.sum())} positifs sur {len(yte)} : un seul dossier de plus ou de moins "
                   "déplace le rappel de plusieurs points. ROC-AUC et PR-AUC ne dépendent pas du seuil.")

        c1, c2 = st.columns(2)
        fig = go.Figure()
        for n, p in proba.items():
            fpr, tpr, _ = roc_curve(yte, p)
            fig.add_scatter(x=fpr, y=tpr, name=f"{n} ({roc_auc_score(yte, p):.2f})")
        fig.add_scatter(x=[0, 1], y=[0, 1], line=dict(dash="dash"), name="Hasard")
        fig.update_layout(title="Courbes ROC", xaxis_title="Taux de faux positifs", yaxis_title="Rappel")
        c1.plotly_chart(fig, width="stretch")

        fig = go.Figure()
        for n, p in proba.items():
            pr, rc, _ = precision_recall_curve(yte, p)
            fig.add_scatter(x=rc, y=pr, name=f"{n} ({average_precision_score(yte, p):.2f})")
        fig.add_hline(y=float(yte.mean()), line_dash="dash", annotation_text="hasard")
        fig.update_layout(title="Courbes précision-rappel", xaxis_title="Rappel", yaxis_title="Précision")
        c2.plotly_chart(fig, width="stretch")

        st.subheader("Analyse d'un modèle")
        name = st.selectbox("Modèle", list(proba), key="eval_model")
        p = proba[name]
        pred = (p >= thr).astype(int)
        a, b = st.columns(2)
        cm = confusion_matrix(yte, pred, labels=[0, 1])
        a.plotly_chart(px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                                 x=["Prédit négatif", "Prédit positif"], y=["Réel négatif", "Réel positif"],
                                 title=f"Matrice de confusion (seuil {thr:.2f})"), width="stretch")
        ths = np.linspace(0.02, 0.98, 49)
        curve = pd.DataFrame({
            "Seuil": ths,
            "Précision": [precision_score(yte, (p >= t).astype(int), zero_division=0) for t in ths],
            "Rappel": [recall_score(yte, (p >= t).astype(int), zero_division=0) for t in ths],
            "F1": [f1_score(yte, (p >= t).astype(int), zero_division=0) for t in ths]})
        fig = px.line(curve, x="Seuil", y=["Précision", "Rappel", "F1"], title="Effet du seuil")
        fig.add_vline(x=thr, line_dash="dash")
        b.plotly_chart(fig, width="stretch")
        st.caption("Baisser le seuil détecte plus de fraudes (rappel ↑) mais génère plus de fausses alertes (précision ↓). "
                   "Le seuil doit être choisi selon le coût d'une enquête inutile face à une fraude manquée, "
                   "et surtout pas en regardant ce jeu de test.")

Appending to app.py


**Cellule 46** : relancer Streamlit

In [49]:
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

**Étape 1 : l’app exporte les modèles et l’API se recharge**

**Cellule A**: réécrit api.py. Il relit le bundle dès qu’il change, donc pas besoin de redémarrer l’API après un réentraînement.

In [50]:
%%writefile api.py
import os
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

BUNDLE_PATH = "bundle.joblib"
_cache = {"mtime": None, "bundle": None}

def get_bundle():
    """Recharge le bundle s'il a changé (réentraînement depuis l'application)."""
    if not os.path.exists(BUNDLE_PATH):
        raise HTTPException(503, "Aucun modèle entraîné : entraîne d'abord les modèles dans l'application.")
    m = os.path.getmtime(BUNDLE_PATH)
    if _cache["mtime"] != m:
        _cache.update(mtime=m, bundle=joblib.load(BUNDLE_PATH))
    return _cache["bundle"]

app = FastAPI(
    title="API de détection de fraude auto",
    description="Prédit la probabilité qu'un dossier de sinistre automobile soit frauduleux.",
    version="1.1",
)

class PredictRequest(BaseModel):
    modele: str = Field("XGBoost", description="Nom du modèle (voir GET /models)")
    seuil: float = Field(0.5, ge=0, le=1, description="Seuil de décision")
    variables: dict = Field(..., description="Valeurs des variables explicatives du dossier")

class PredictResponse(BaseModel):
    modele: str
    probabilite_fraude: float
    seuil: float
    decision: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/models")
def list_models():
    b = get_bundle()
    return {"modeles": list(b["models"]), "variables_attendues": b["features"]}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    b = get_bundle()
    models, features = b["models"], b["features"]
    if req.modele not in models:
        raise HTTPException(404, f"Modèle inconnu : {req.modele}")
    missing = [f for f in features if f not in req.variables]
    if missing:
        raise HTTPException(422, f"Variables manquantes : {missing}")
    X = pd.DataFrame([req.variables])[features]
    p = float(models[req.modele].predict_proba(X)[0, 1])
    return PredictResponse(
        modele=req.modele,
        probabilite_fraude=round(p, 4),
        seuil=req.seuil,
        decision="À investiguer" if p >= req.seuil else "Traitement standard",
    )

Overwriting api.py


**Cellule B** : patch d’app.py. Après chaque entraînement, l’app écrit le bundle, d’abord dans un fichier temporaire pour éviter que l’API lise un fichier à moitié écrit.

In [51]:
s = open("app.py").read()

a = "import streamlit as st\n"
assert a in s
s = s.replace(a, "import os\nimport joblib\nimport streamlit as st\n", 1)

a = 'mode=mode, sig=make_sig(P))'
b = (a + '\n'
     '            joblib.dump({"models": fitted, "features": P["features"]}, "bundle.tmp")\n'
     '            os.replace("bundle.tmp", "bundle.joblib")')
assert a in s
s = s.replace(a, b, 1)

open("app.py", "w").write(s)
print("ok")

ok


**Cellule C** : relance l’API et Streamlit.

In [53]:
proc.terminate()                      # ancien uvicorn lancé à la cellule 30
!pkill -f streamlit
proc = subprocess.Popen(["uvicorn", "api:app", "--port", "8000"])
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

In [56]:
requests.get("http://127.0.0.1:8000/models").json()

{'modeles': ['Régression logistique',
  'Arbre de décision',
  'Random Forest',
  'XGBoost',
  'KNN'],
 'variables_attendues': ['months_as_customer',
  'age',
  'policy_deductable',
  'policy_annual_premium',
  'umbrella_limit',
  'capital-gains',
  'capital-loss',
  'incident_hour_of_the_day',
  'number_of_vehicles_involved',
  'bodily_injuries',
  'witnesses',
  'injury_claim',
  'property_claim',
  'vehicle_claim',
  'auto_year',
  'policy_age_days',
  'incident_dow',
  'policy_state',
  'policy_csl',
  'insured_sex',
  'insured_education_level',
  'insured_occupation',
  'insured_hobbies',
  'insured_relationship',
  'incident_type',
  'collision_type',
  'incident_severity',
  'authorities_contacted',
  'incident_state',
  'incident_city',
  'property_damage',
  'police_report_available',
  'auto_make',
  'auto_model']}

**Étape 2 : l’onglet « 6 · API »**

**Cellule D** : déclare le 5ᵉ onglet

In [57]:
s = open("app.py").read()
a = 'tab1, tab2, tab3, tab4 = st.tabs(["1 · Données", "2 · Préparation", "3 · Modélisation", "4 · Évaluation"])'
b = 'tab1, tab2, tab3, tab4, tab5 = st.tabs(["1 · Données", "2 · Préparation", "3 · Modélisation", "4 · Évaluation", "6 · API"])'
assert a in s
open("app.py", "w").write(s.replace(a, b))
print("ok")

ok


**Cellule E** : ajoute le contenu de l’onglet.

In [58]:
%%writefile -a app.py

import requests

with tab5:
    st.subheader("Prédiction via l'API FastAPI")
    api_url = st.text_input("URL de l'API", "http://127.0.0.1:8000")

    if "prep" not in st.session_state:
        st.info("Prépare d'abord les données dans l'onglet 2 · Préparation.")
        st.stop()
    try:
        info = requests.get(f"{api_url}/models", timeout=5)
        info.raise_for_status()
        info = info.json()
    except Exception as e:
        st.error(f"API injoignable ou sans modèle entraîné : {e}")
        st.stop()

    P = st.session_state["prep"]
    feats = info["variables_attendues"]
    if feats != list(P["features"]):
        st.warning("Les variables de l'API ne correspondent plus à la préparation actuelle : "
                   "relance « Entraîner et comparer » dans l'onglet 3.")
        st.stop()

    Xtr, Xte, yte = P["X_train"], P["X_test"], P["y_test"]
    st.caption("Le formulaire est pré-rempli avec un dossier du jeu de test ; tu peux modifier les valeurs.")
    pos = int(st.number_input("Dossier du jeu de test (n° de ligne)", 0, len(Xte) - 1, 0))
    row = Xte.iloc[pos]
    st.write(f"Vérité terrain de ce dossier : **{'positif' if yte.iloc[pos] == 1 else 'négatif'}**")

    values = {}
    cols = st.columns(3)
    for i, f in enumerate(feats):
        box, key = cols[i % 3], f"api_{f}_{pos}"
        if f in P["num"]:
            v = row[f]
            v = Xtr[f].median() if pd.isna(v) else v
            if pd.api.types.is_integer_dtype(Xtr[f]):
                values[f] = box.number_input(f, value=int(v), step=1, key=key)
            else:
                values[f] = box.number_input(f, value=float(v), format="%.2f", key=key)
        else:
            opts = sorted(Xtr[f].astype(str).unique())
            cur = str(row[f])
            values[f] = box.selectbox(f, opts, index=opts.index(cur) if cur in opts else 0, key=key)

    c1, c2 = st.columns(2)
    names = info["modeles"]
    modele = c1.selectbox("Modèle", names, index=names.index("XGBoost") if "XGBoost" in names else 0, key="api_model")
    seuil = c2.slider("Seuil de décision", 0.05, 0.95, 0.50, 0.05, key="api_thr")

    if st.button("Prédire via l'API", type="primary", key="api_go"):
        r = requests.post(f"{api_url}/predict",
                          json={"modele": modele, "seuil": seuil, "variables": values}, timeout=30)
        if r.ok:
            out = r.json()
            m1, m2 = st.columns(2)
            m1.metric("Probabilité", f"{out['probabilite_fraude']:.1%}")
            m2.metric("Décision", out["decision"])
            st.progress(min(float(out["probabilite_fraude"]), 1.0))
        else:
            st.error(f"Erreur {r.status_code} : {r.json().get('detail', r.text)}")

Appending to app.py


**Cellule F** : relance Streamlit (l’API tourne déjà)

In [63]:
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

**Étape 3 : préparer le dépôt GitHub**

In [60]:
s = open("app.py").read()
a = "import os\nimport joblib\nimport streamlit as st\n"
b = '''import os, sys, subprocess
import joblib
import requests
import streamlit as st

@st.cache_resource
def start_api():
    """Lance l'API FastAPI en arrière-plan si elle ne tourne pas déjà."""
    try:
        requests.get("http://127.0.0.1:8000/health", timeout=2)
        return None
    except Exception:
        return subprocess.Popen([sys.executable, "-m", "uvicorn", "api:app", "--port", "8000"])

start_api()
'''
assert a in s
open("app.py", "w").write(s.replace(a, b, 1))
print("ok")

ok


**Cellule H** : génère un requirements.txt avec les versions installées dans Colab. Ton app utilise width="stretch", qui n’existe que dans les versions récentes de Streamlit, donc il faut les fixer.

In [61]:
import subprocess
pk = ["streamlit", "pandas", "numpy", "scikit-learn", "xgboost", "plotly", "joblib", "requests", "fastapi", "uvicorn", "scipy"]
frz = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
keep = [l for l in frz if l.split("==")[0].lower() in pk]
open("requirements.txt", "w").write("\n".join(keep) + "\n")
print(open("requirements.txt").read())

fastapi==0.141.1
joblib==1.6.0
numpy==2.1.3
pandas==2.2.3
plotly==5.24.1
requests==2.32.4
scikit-learn==1.6.1
scipy==1.16.3
streamlit==1.65.0
uvicorn==0.54.0
xgboost==3.4.1



**Cellule I** : télécharge les fichiers.

In [62]:
from google.colab import files
for f in ["app.py", "core.py", "api.py", "requirements.txt"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>